# Notebook 02 — Baseline Models
**DSA 3900 | Maternal Health Risk Prediction — Kenya**

Logistic Regression and Decision Tree baselines. Establishes performance
floor for comparison with optimised models.

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix)

PROC  = Path('../../01_Data/processed')
MOUT  = Path('../../03_Models_Saved')
FOUT  = Path('../../04_Outputs/figures')
ROUT  = Path('../../04_Outputs/results')
for p in [MOUT, FOUT, ROUT]: p.mkdir(parents=True, exist_ok=True)

print('Libraries loaded ✓')

## 1. Load Splits

In [ ]:
X_train = pd.read_csv(PROC / 'X_train.csv')
X_val   = pd.read_csv(PROC / 'X_val.csv')
X_test  = pd.read_csv(PROC / 'X_test.csv')
y_train = pd.read_csv(PROC / 'y_train.csv').squeeze()
y_val   = pd.read_csv(PROC / 'y_val.csv').squeeze()
y_test  = pd.read_csv(PROC / 'y_test.csv').squeeze()

print(f'Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}')

## 2. Evaluation Helper

In [ ]:
CLASS_NAMES = ['Low','Moderate','High']

def evaluate_model(model, X, y, model_name, split='Val'):
    """Compute metrics, print report, save confusion matrix PNG."""
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)

    acc   = accuracy_score(y, y_pred)
    macro = f1_score(y, y_pred, average='macro')
    high  = f1_score(y, y_pred, average=None)[2]
    auc   = roc_auc_score(y, y_proba, multi_class='ovr', average='macro')

    print(f'\n=== {model_name} ({split}) ===')
    print(classification_report(y, y_pred, target_names=CLASS_NAMES))

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{model_name} — {split} Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    fname = f'{model_name.replace(" ","_")}_{split}_cm.png'
    fig.savefig(FOUT / fname, dpi=150)
    plt.show()

    return {'model': model_name, 'accuracy': round(acc,4),
            'macro_f1': round(macro,4), 'high_risk_f1': round(high,4),
            'roc_auc': round(auc,4)}

print('evaluate_model() defined ✓')

## 3. Logistic Regression

In [ ]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
joblib.dump(lr, MOUT / 'lr_model.pkl')

lr_res = evaluate_model(lr, X_val, y_val, 'Logistic Regression')

### Interpretation — Logistic Regression
*(Paste your output here after running the cell)*

## 4. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)
joblib.dump(dt, MOUT / 'dt_model.pkl')

dt_res = evaluate_model(dt, X_val, y_val, 'Decision Tree')

### Interpretation — Decision Tree
*(Paste your output here after running the cell)*

## 5. Baseline Comparison

In [ ]:
results = pd.DataFrame([lr_res, dt_res])
results.to_csv(ROUT / 'baseline_results.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [('macro_f1','Macro F1'), ('high_risk_f1','High-Risk F1'), ('roc_auc','ROC-AUC')]

for ax, (col, label) in zip(axes, metrics):
    bars = ax.bar(results['model'], results[col], color=['#4C72B0','#DD8452'])
    ax.set_title(label); ax.set_ylim(0, 1)
    ax.set_xticklabels(results['model'], rotation=15, ha='right')
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.suptitle('Baseline Model Comparison (Validation Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FOUT / 'baseline_comparison.png', dpi=150)
plt.show()
print(results.to_string(index=False))

### Interpretation — Baseline Comparison
*(Paste your output here after running the cell)*

## ✅ Baseline Models Complete
Models saved: `lr_model.pkl`, `dt_model.pkl`  
Results saved: `baseline_results.csv`  
→ Proceed to Notebook 03 for Optuna-optimised classical models.